In [67]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os

def calculate_vif(df):
    """Calculate VIF for a DataFrame, iteratively removing features with VIF > 100."""
    df_numeric = df.select_dtypes(include=['number'])  # Ensure only numeric columns
    if df_numeric.empty:
        print("No numeric columns available for VIF calculation.")
        return None

    removed_features = []
    while True:
        vif_data = pd.DataFrame()
        vif_data["Feature"] = df_numeric.columns
        vif_data["VIF"] = [variance_inflation_factor(df_numeric.values, i) for i in range(df_numeric.shape[1])]

        high_vif_features = vif_data[vif_data["VIF"] > 100]

        if high_vif_features.empty:
            break  # Exit loop if no feature has VIF > 100

        # Remove the feature with the highest VIF
        feature_to_remove = high_vif_features.loc[high_vif_features["VIF"].idxmax(), "Feature"]
        print(f"Removing feature: {feature_to_remove}, VIF: {high_vif_features.loc[high_vif_features['VIF'].idxmax(), 'VIF']:.2f}")
        df_numeric = df_numeric.drop(columns=[feature_to_remove])
        removed_features.append(feature_to_remove)

    if removed_features:
        print("\nRemoved Features:")
        for feature in removed_features:
            print(f"  {feature}")

    vif_data = vif_data.set_index("Feature")  # Set Features as row index
    return vif_data

def csvs_to_dfs_per_folder(root_directory):
    """Reads CSV files in each folder under the root directory, storing them in a dictionary."""
    all_dataframes = {}
    all_global_names = {}

    for folder_name in os.listdir(root_directory):
        folder_path = os.path.join(root_directory, folder_name)

        if os.path.isdir(folder_path):
            folder_dataframes = {}

            for filename in os.listdir(folder_path):
                if filename.endswith(".csv"):
                    filepath = os.path.join(folder_path, filename)
                    csv_base_name = os.path.splitext(filename)[0].lower().replace(" ", "_")
                    df_name = csv_base_name  # Removed folder prefix

                    try:
                        df = pd.read_csv(filepath)
                        folder_dataframes[df_name] = df
                        all_dataframes[df_name] = df
                        print(f"Successfully read: {filename} from {folder_name}")
                    except Exception as e:
                        print(f"Error reading {filename} from {folder_name}: {e}")

            all_global_names[folder_name] = list(folder_dataframes.keys())

    return all_global_names, all_dataframes

def concat_year(year, df_names, dataframes):
    """Concatenates DataFrames for a given year, keeping clean column names."""
    selected_dfs = []

    for name in df_names:
        df = dataframes[name]
        if year in df.columns:
            selected_df = df[["PROVINCE / LGU", year]].copy()
            selected_df.columns = ["PROVINCE / LGU", name]  # Keep only base name
            selected_dfs.append(selected_df.set_index("PROVINCE / LGU"))

    if selected_dfs:
        result_df = pd.concat(selected_dfs, axis=1)
        return result_df
    else:
        print(f"No valid data found for the year {year}.")
        return None

# Define directories
root_directory = "CMCI Data Raw"  # Change as needed
vif_output_directory = "CMCI VIF"

# Create output directory if not exists
os.makedirs(vif_output_directory, exist_ok=True)

# Read CSVs from folders
all_df_names, all_dfs = csvs_to_dfs_per_folder(root_directory)

# Process each folder
for folder_name, folder_df_names in all_df_names.items():
    folder_dfs = {name: all_dfs[name] for name in folder_df_names}

    if folder_df_names:
        df_2023 = concat_year("2023", folder_df_names, folder_dfs)
        if df_2023 is not None:
            output_filename = f"CMCI_2023_{folder_name}.csv"
            df_2023.to_csv(output_filename, encoding="utf-8-sig", index=True)  # Save with cleaned column names
            print(f"Concatenated data for {folder_name} saved to {output_filename}")

            # Calculate VIF, ensuring "PROVINCE / LGU" is ignored
            vif_result = calculate_vif(df_2023.drop("PROVINCE / LGU", axis=1, errors='ignore'))
            if vif_result is not None:
                vif_filename = f"VIF_CMCI_2023_{folder_name}.csv"
                vif_filepath = os.path.join(vif_output_directory, vif_filename)
                vif_result.to_csv(vif_filepath, index=True)  # Saves with features as row index
                print(f"VIF for {folder_name} saved to {vif_filepath}")
    else:
        print(f"No 2023 data to concatenate for {folder_name}.")

Successfully read: Active Establishments in the Locality.csv from Economic Dynamism
Successfully read: Cost of Doing Business.csv from Economic Dynamism
Successfully read: Cost of Living.csv from Economic Dynamism
Successfully read: Employment Generation.csv from Economic Dynamism
Successfully read: Financial Deepening.csv from Economic Dynamism
Successfully read: Local Economy Growth.csv from Economic Dynamism
Successfully read: Local Economy Size.csv from Economic Dynamism
Successfully read: Presence of Business and Professional Organizations.csv from Economic Dynamism
Successfully read: Productivity.csv from Economic Dynamism
Successfully read: Safety Compliant Business.csv from Economic Dynamism
Successfully read: Capacity of Health Services.csv from Government Efficiency
Successfully read: Capacity of School Services.csv from Government Efficiency
Successfully read: Capacity to Generate Local Resource.csv from Government Efficiency
Successfully read: Compliance to ARTA Citizens Ch

c:\Users\Leibniz\anaconda3\lib\site-packages\statsmodels\stats\outliers_influence.py:195: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [68]:
import os
import pandas as pd
import re

folder_path = "CMCI VIF"
csv_files = [f for f in os.listdir(folder_path) if f.startswith("VIF_CMCI_2023_") and f.endswith(".csv")]

def to_snake_case(name):
    name = name.lower()
    name = re.sub(r'\s+', '_', name)
    name = re.sub(r'[^a-z0-9_]', '', name)
    return name

vif_dfs = {}
for file in csv_files:
    var_name = file.replace("VIF_CMCI_2023_", "").replace(".csv", "")
    df_name = f"df_VIF_{to_snake_case(var_name)}"

    # Read CSV
    df = pd.read_csv(os.path.join(folder_path, file))

    globals()[df_name] = df
    vif_dfs[df_name] = df

    # Print DataFrame
    print(f"\n### {df_name} ###\n")
    print(df)

# Function to filter VIF DataFrames and print selected features
def filter_vif_and_print(vif_dct):
    selected_features = set()

    for vif_df_name, vif_df in vif_dct.items():
        filtered_rows = vif_df[vif_df["VIF"] < 10]["Feature"].tolist()
        selected_features.update(filtered_rows)
        print(len(filtered_rows), vif_df_name, filtered_rows)

    return list(selected_features)

# Get selected features (VIF < 10)
selected_features = filter_vif_and_print(vif_dfs)
print(selected_features)


### df_VIF_economic_dynamism ###

                                             Feature        VIF
0              active_establishments_in_the_locality  11.382224
1                             cost_of_doing_business  21.478783
2                                     cost_of_living  19.487497
3                              employment_generation   9.472916
4                                financial_deepening   9.180801
5                               local_economy_growth   2.002097
6                                 local_economy_size   7.477743
7  presence_of_business_and_professional_organiza...   2.733491
8                                       productivity  13.954281
9                          safety_compliant_business  10.562483

### df_VIF_government_efficiency ###

                                 Feature        VIF
0            capacity_of_health_services   3.686907
1            capacity_of_school_services   5.639028
2    capacity_to_generate_local_resource   3.863595
3      complia

In [86]:
import os
import pandas as pd


# Combine all 2023 DataFrames
data_folder = os.getcwd()  # Change if needed
csv_2023_files = [f for f in os.listdir(data_folder) if f.startswith("CMCI_2023_") and f.endswith(".csv")]
csv_2023_files = [item for item in csv_2023_files if item not in ["CMCI_2023_Combined.csv", "CMCI_2023_Filtered.csv", "CMCI_2023_Full.csv", "CMCI_2023_Shortlist.csv"]]
print(csv_2023_files)

combined_2023_df = pd.DataFrame()
filtered_2023_df = pd.DataFrame()

for file in csv_2023_files:
    try:
        df = pd.read_csv(file)
    except pd.errors.EmptyDataError:
        print(f"Warning: Empty file {file}. Skipping.")
        continue  # Skip to the next file
    except FileNotFoundError:
        print(f"Warning: File not found: {file}. Skipping.")
        continue
    except Exception as e:
        print(f"Warning: Error reading file {file}: {e}. Skipping.")
        continue

    # Extract folder name from the filename
    folder_name = file.replace("CMCI_2023_", "").replace(".csv", "")

    if combined_2023_df.empty:
        combined_2023_df = df
    else:
        combined_2023_df = combined_2023_df.merge(df, on="PROVINCE / LGU", how="outer")

    # Keep only selected features (renamed)
    filtered_cols = ["PROVINCE / LGU"] + [col for col in df.columns if col != "PROVINCE / LGU" and col in selected_features]

    filtered_df = df[filtered_cols]

    if filtered_2023_df.empty:
        filtered_2023_df = filtered_df
    else:
        filtered_2023_df = filtered_2023_df.merge(filtered_df, on="PROVINCE / LGU", how="outer")

filtered_len = len(filtered_2023_df.columns)
filtered_len_removed = len(combined_2023_df.columns) - filtered_len
print(f"Rows removed due to VIF filtering: {filtered_len_removed}")

# Apply null threshold to filtered_2023_df
if not filtered_2023_df.empty:
    null_threshold = len(filtered_2023_df) * (0.20)
    cols_to_drop = filtered_2023_df.columns[filtered_2023_df.isnull().sum() > null_threshold]
    if len(cols_to_drop) > 0:
        print(f"Removed columns with >5% null values from filtered_2023_df: {list(cols_to_drop)}")
        filtered_2023_df = filtered_2023_df.drop(columns=cols_to_drop)
null_len_removed = filtered_len - len(filtered_2023_df.columns)
print(f"Rows removed due to null removcal: {null_len_removed}")

# Save the combined DataFrames
combined_2023_df.to_csv("CMCI_2023_Combined.csv", index=False)
filtered_2023_df.to_csv("CMCI_2023_Filtered.csv", index=False)

print("\n### Combined 2023 DataFrame Saved: CMCI_2023_combined.csv ###\n")
print(combined_2023_df.shape)

print("\n### Filtered 2023 DataFrame Saved: CMCI_2023_filtered.csv ###\n")
print(filtered_2023_df.shape)

['CMCI_2023_Economic Dynamism.csv', 'CMCI_2023_Government Efficiency.csv', 'CMCI_2023_Infrastructure.csv', 'CMCI_2023_Innovation.csv', 'CMCI_2023_Overall.csv', 'CMCI_2023_Resilience.csv']
Rows removed due to VIF filtering: 30
Removed columns with >5% null values from filtered_2023_df: ['capacity_of_health_services', 'capacity_of_school_services', 'capacity_to_generate_local_resource', 'peace_and_order', 'recognition_of_performance', 'social_protection', 'accommodation_capacity', 'financial_technology_capacity', 'information_technology_capacity', 'lgu_investment', 'road_network', 'transportation_vehicles', 'availability_of_basic_internet_service', 'innovation_financing__r&d_expenditures_allotment', 'intellectual_property_registration', 'new_technology', 'online_payment_facilities', 'start_up_and_innovation_facilities', 'stem_graduates', 'budget_for_drrmp', 'employed_population']
Rows removed due to null removcal: 21

### Combined 2023 DataFrame Saved: CMCI_2023_combined.csv ###

(54, 57

In [77]:
def calculate_vif_for_filtered_csv(csv_filename):
    """Calculates VIF for the CMCI_2023_filtered.csv file."""
    try:
        df = pd.read_csv(csv_filename)
        # remove non-numeric column
        df_numeric = df.drop("PROVINCE / LGU", axis=1, errors='ignore')
        vif_result = calculate_vif(df_numeric)

        if vif_result is not None:
            print(f"\nVIF Results for {csv_filename}:")
            print(vif_result)
        else:
            print(f"VIF calculation failed for {csv_filename}.")

    except FileNotFoundError:
        print(f"File not found: {csv_filename}")
    except pd.errors.EmptyDataError:
        print(f"Empty data in {csv_filename}")
    except Exception as e:
        print(f"Error processing {csv_filename}: {e}")

# Example usage:
csv_filename = "CMCI_2023_Filtered.csv"  # Replace with the actual filename
calculate_vif_for_filtered_csv(csv_filename)


VIF Results for CMCI_2023_Filtered.csv:
                                                         VIF
Feature                                                     
employment_generation                               4.165096
financial_deepening                                 3.331748
local_economy_growth                                1.747077
local_economy_size                                  3.039479
presence_of_business_and_professional_organizat...  2.457561
